# Bindary phase diagram
## Review of Completely Miscible Alloys


For constructing the phase diagram, we need to account for the difference in the Gibbs free energy between the solid and liquid phases for the pure components A and B. We will use a simplified thermodynamics model for the pure components that assumes that the difference in entropy and enthalpy between the solid and liquid does not depend on temperature.

We will then use the tangent intercept construction to determine the chemical potentials for the components A and B in each phase:
$$
\mu_A = G(x_B) - x_B \frac{dG(x_B)}{d x_B}
$$
and
$$
\mu_B = G(x_B) + (1-x_B) \frac{dG(x_B)}{d x_B}.
$$

In equilibrium, the chemical potentials of a every component are the same in each phase:
$$
\mu_A^S = \mu_A^L
$$
and
$$
\mu_B^S = \mu_B^L.
$$

The mathematical solution to the chemical equilibrium condition is the common tangent construction.

### Preliminaries
Import libraries.

In [ ]:
# Import NumPy numerical package
import numpy as np

# Import SciPy
import scipy as scipy
from scipy.optimize import fsolve, newton

# Enable displaying of math output such as display(Math(r'Area: {}m^2 \\ Volume: {}m^3'.format(a, round(b,2), A, V)))
from IPython.display import display, Math

# plotting package
import matplotlib.pyplot as plt

import matplotlib


### Thermodynamic data

Set the thermodynamic constant and define the thermodynamic functions for the regular solution model.

We will use a simple model for the Gibbs free energy of the solid (S) and liquid (L) phases and describe both by the ideal solution model:
$$
\Delta G_\mathrm{mix}(x_B) = RT \left [ x_B \ln x_B +(1−x_B) \ln(1−x_B) \right ].
$$
Here, $R$ is the gas constant (8.314 J/mol/K), $T$ the absolute temperature, $x_B$ is the mole fraction of component B.

In [ ]:
# Gas constant
R = 8.314 # J/mol/K

# Return the ideal entropy of mixing as a function of temperature and mole fraction 
def get_S_id(T,x):
    S_id = -R * (x * np.log(x) + (1-x)*np.log(1-x))
    return S_id

# Return the enthalpy for a regular solution
def get_H_rs(T, a, x):
    H_rs = a * x * (1-x)
    return H_rs

# Return the Gibbs free energy of mixing for a regular solution
def get_G_rs(T, a, x):
    S_id = get_S_id(T,x)
    H_rs = get_H_rs(T, a, x)
    G_rs = H_rs - T * S_id
    return G_rs
    

In [ ]:
# Array of temperatures from 0 to 1000 K
Tv = np.linspace(0,1000,20)

# Difference in Gibbs free energy between the liquid and solid phase for the pure components A and B
# Assume that the Gibbs free energy difference is simply linear in temperature
def get_Gref(T, a, b):
    return a*T-b
    
aA = 24
bA = 10e3

aB = 16
bB = 12e3

def GrefA_func(T):
    return get_Gref(T, aA, bA)

def GrefB_func(T):
    return get_Gref(T, aB, bB)

TmA = newton(GrefA_func, 500)
TmB = newton(GrefB_func, 500)
print("Melting point\n  of component A = ", TmA, "\n  of component B = ", TmB)


In [ ]:
# Array of compositions from 0 to 1
xv = np.linspace(0.0001,0.9999,500)

# Set the temperature
T = 300 # K

# Gibbs free energy of mixing for the ideal solition model (a0=0)
Gmix_S = get_G_rs(T, 0.0, xv)
Gmix_L = get_G_rs(T, 0.0, xv)

# Reference Gibbs free energies for pure components
GrefA = get_Gref(T, aA, bA)
GrefB = get_Gref(T, aB, bB)

# Return the Gibbs free energy accounting for the reference states for A and B
# We select the liquid phase as the reference state for both pure components
def get_G_S(T, x):
    return get_G_rs(T, 0, x) + get_Gref(T, aA, bA)*(1-x) + get_Gref(T, aB, bB)*x

def get_G_L(T, x):
    return get_G_rs(T, 0, x)

G_S = get_G_S(T, xv)
G_L = get_G_L(T, xv)

# Plot Gibbs free energy for both phases
fig, ax = plt.subplots(1, 1, figsize=(5,5))
fig.tight_layout()

ax.plot(xv, G_S , 'b-', linewidth=3.0, label='$G^\\mathrm{S}$')
ax.plot(xv, G_L, 'r-', linewidth=3.0, label='$G^\\mathrm{L}$')
ax.plot([0, 1], [0, 0], 'k--', linewidth=2.0)
ax.set_xlim([0, 1])
ax.set_xlabel('$x_\\mathrm{B}$')
ax.set_ylabel('Gibbs free energy (J/mol)')
ax.legend(prop={'size': 18})

fig.tight_layout()
plt.show()

The choice of reference state is arbitrary and does not change the thermodynamics. In the above example, we use the liquid phase as the reference state for pure component A and B. Hence, the Gibbs free energy of the liquid phase goes to zero for $x_B = 0$ and $x_B = 1$.

We can change the reference state for pure component A to be the solid phase, such that we subtract A's reference energy for both solid and liquid phases.

In [ ]:
# Return the Gibbs free energy accounting for the reference states for A and B
# We select the liquid phase as the reference state for both pure components
def get_G_S(T, x):
    return get_G_rs(T, 0, x) + get_Gref(T, aB, bB)*x

def get_G_L(T, x):
    return get_G_rs(T, 0, x) - get_Gref(T, aA, bA)*(1-x)

G_S = get_G_S(T, xv)
G_L = get_G_L(T, xv)

# Plot Gibbs free energy for both phases
fig, ax = plt.subplots(1, 1, figsize=(5,5))
fig.tight_layout()

ax.plot(xv, G_S, 'b-', linewidth=3.0, label='$G^\\mathrm{S}$')
ax.plot(xv, G_L, 'r-', linewidth=3.0, label='$G^\\mathrm{L}$')
ax.plot([0, 1], [0, 0], 'k--', linewidth=2.0)
ax.set_xlim([0, 1])
ax.set_xlabel('$x_\\mathrm{B}$')
ax.set_ylabel('Gibbs free energy (J/mol)')
ax.legend(prop={'size': 18})

fig.tight_layout()
plt.show()

### Tangent intercept construction for the chemical potentials of A and B

Now we calculate the chemical potentials of components A and B for arbitrary mole fractions using the tangent intercept construction:
$$
\mu_A = G(x_B) - x_B \frac{dG(x_B)}{d x_B}
$$
and
$$
\mu_B = G(x_B) + (1-x_B) \frac{dG(x_B)}{d x_B}.
$$

In [ ]:
# Tangent to the Gibbs free energy curve

# Set the temperature
T = 300 # K


G_S = get_G_S(T, xv)
G_L = get_G_L(T, xv)

# Tangent intercept construction of chemical potentials of components A and B

# Select a mole fraction
x = 0.4

# Calculate the derivative of G_S with respect to x
dx = 0.0001
dG_Sdx = (get_G_S(T, x+dx/2) - get_G_S(T, x-dx/2))/dx
mu_A = get_G_S(T, x) - x*dG_Sdx
mu_B = get_G_S(T, x) + (1-x)*dG_Sdx

# Plot Gibbs free energy for both phases
fig, ax = plt.subplots(1, 1, figsize=(5,5))
fig.tight_layout()

ax.plot(xv, G_S, 'b-', linewidth=3.0, label='$G^\\mathrm{S}$')
ax.plot(xv, G_L, 'r-', linewidth=3.0, label='$G^\\mathrm{L}$')
ax.plot([0, 1], [mu_A, mu_B], 'b--', linewidth=2.0)
ax.plot([0, 1], [0, 0], 'k--', linewidth=2.0)

ax.annotate('$\\mu_A$', xy=(-0.08, mu_A), color='b', xycoords='data', annotation_clip=False)
ax.annotate('$\\mu_B$', xy=(1.02,mu_B), color='b', xycoords='data', annotation_clip=False)

ax.set_xlim([0, 1])
ax.set_xlabel('$x_{B}$')
ax.set_ylabel('Gibbs free energy (J/mol)')
ax.legend(prop={'size': 18})
fig.tight_layout()
plt.show()

### Calculate a common tangent

Finally, we will calculate the common tangent to determine the chemical equilibrium between the solid and liquid phase. We will use the Python function `fsolve` to determine the mole fraction $x_B$ in the solid and liquid phase such that the chemical potentials in each phase are equal:
$$
\mu_A^S = \mu_A^L
$$
and
$$
\mu_B^S = \mu_B^L.
$$

---
The two cells below demonstrate how to use `fsolve` in `scipy.optimize`.

```fsolve(f, x0)```

 Means find the root of the defined function `f` using `x0` as the initial guess.

Solve 

$$x^2-4=0.$$

In [ ]:
# define a function, this function needs an input x
def f(x):
    return x**2 - 4

# initial guess
x0 = 1.0

root0 = fsolve(f, x0)
print(root0)


x1 = -1.0
root1 = fsolve(f, x1)
print(root1)

Solve 
$$
\begin{cases}
x^2 + y^2 = 5 \\
x - y = 1
\end{cases}
$$

In [ ]:
def F(p):
    x, y = p
    return [x**2 + y**2 - 5, x - y - 1]

sol = fsolve(F, [1, 1])
print(sol)

---
Back to the Gibbs free energy and chemical potential

In [ ]:
# Set the temperature
T = 500 # K

G_S = get_G_S(T, xv)
G_L = get_G_L(T, xv)

# Return the chemical potentials of components A and B for the solid phase
def get_mu_S(T, x):
    dx = 0.0001
    dGdx = (get_G_S(T, x+dx/2) - get_G_S(T, x-dx/2))/dx
    mu_A = get_G_S(T, x) - x*dGdx
    mu_B = get_G_S(T, x) + (1-x)*dGdx
    return mu_A, mu_B

# Return the chemical potentials of components A and B for the liquid phase
def get_mu_L(T, x):
    dx = 0.0001
    dGdx = (get_G_L(T, x+dx/2) - get_G_L(T, x-dx/2))/dx
    mu_A = get_G_L(T, x) - x*dGdx
    mu_B = get_G_L(T, x) + (1-x)*dGdx
    return mu_A, mu_B

# Find the composition x such that mu_A^S = mu_A^L and mu_B^S = mu_B^L
def commontangent(p):
    x_S, x_L = p
    mu_A_S, mu_B_S = get_mu_S(T, x_S)
    mu_A_L, mu_B_L = get_mu_L(T, x_L)
    return (mu_A_S-mu_A_L), (mu_B_S-mu_B_L)

# May need to ensure that we use a reasonable starting guess
x_S, x_L =  fsolve(commontangent, (0.7, 0.3))

mu_A_S, mu_B_S = get_mu_S(T,x_S)
mu_A_L, mu_B_L = get_mu_L(T,x_L)

print('Common tangent construction:')
display(Math(r' x_S = {}, x_L = {}'.format(round(x_S,3), round(x_L,3))))

# Plot Gibbs free energy for both phases
fig, ax = plt.subplots(1, 1, figsize=(5,5))
fig.tight_layout()

ax.plot(xv, G_S, 'b-', linewidth=3.0, label='$G^\\mathrm{S}$')
ax.plot(xv, G_L, 'r-', linewidth=3.0, label='$G^\\mathrm{L}$')
ax.plot([0, 1], [mu_A_S, mu_B_S], 'k--', linewidth=3.0)
ax.plot([0, 1], [0, 0], 'k--', linewidth=2.0)

ax.annotate('$\\mu_A$', xy=(-0.08, mu_A_S), color='k', xycoords='data', annotation_clip=False)
ax.annotate('$\\mu_B$', xy=(1.02, mu_B_S), color='k', xycoords='data', annotation_clip=False)

ymin, ymax = ax.get_ylim()
ax.plot([x_S, x_S], [get_G_S(T, x_S), ymin], 'k-', linewidth=1.0)
ax.plot([x_L, x_L], [get_G_L(T, x_L), ymin], 'k-', linewidth=1.0)

ax.set_xlim([0, 1])
ax.set_ylim([ymin, ymax])
ax.set_xlabel('$x_\\mathrm{B}$')
ax.set_ylabel('Gibbs free energy (J/mol)')
ax.legend(prop={'size': 18})
fig.tight_layout()
plt.show()

---
## Convex Hull method

While solving the roots where the two phases have identical chemical potentials is mathematically rigorous, it can become difficult when the free energy expressions are complex or highly nonlinear. In such cases, a geometric approach based on the convex hull provides a robust and intuitive alternative.

The idea is to consider a set of discrete points representing the free energy curve, 
$G(x)$, as a function of composition. By constructing the lower convex hull of these points in the $(x,G)$ space, we obtain the equilibrium free energy surface. This lower envelope removes any concave (unstable) regions of the original curve.

In this construction, any straight line segment that appears on the lower convex hull corresponds to a common tangent line. The two endpoints of this segment represent the equilibrium compositions of the coexisting phases, $x_\alpha$ and $x_\beta$. Physically, this means that the system lowers its free energy by separating into two phases rather than remaining as a single homogeneous phase.

Mathematically, the convex hull enforces the condition that the free energy is a convex function. Geometrically, this is equivalent to replacing concave regions with straight lines. Thermodynamically, this is equivalent to satisfying the equal chemical potential condition:
$\mu_A^\alpha = \mu_A^\beta$, $\mu_B^\alpha = \mu_B^\beta$.

Thus, the convex hull method, the common tangent construction, and the equality of chemical potentials are all equivalent ways of determining phase equilibrium.

In practice, this method is particularly useful for numerical or noisy data, where analytical derivatives are difficult to obtain. It also extends naturally to higher dimensions, such as ternary systems, where the convex hull in $(x_A, x_B, G)$ space yields common tangent planes and tie triangles.

In the cell below, we use discrete points as the composition and Gibbs free energy data.

In [ ]:
xp = np.linspace(0.0001,0.9999,61)

xp_s = xp
xp_l = xp

Gp_S = get_G_S(T, xp_s)
Gp_L = get_G_L(T, xp_l)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(xp_s, Gp_S, c='b', s=2, label='solid')
ax.scatter(xp_l, Gp_L, c='r', s=2, label='liquid')
ax.legend()

These discrete points are collected to a set of data points. 

In [ ]:
xt1 = np.vstack([xp_s.reshape(-1,1), xp_l.reshape(-1,1)])
yt1 = np.vstack([Gp_S.reshape(-1,1), Gp_L.reshape(-1,1)])

pts = np.column_stack([xt1, yt1])

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(pts[:,0], pts[:,1], c='k', s=2, label='solid')

### lower convex hull

#### Finding the Lower Envelope: Algorithm

To determine the equilibrium free-energy curve from a set of discrete points $(xi,Gi)$, we construct the lower convex envelope. This lower envelope is the lowest convex curve that contains all the data points. In thermodynamics, it represents the stable equilibrium free energy, while any concave portion of the original curve corresponds to unstable or metastable states.

A simple and efficient way to find the lower envelope is to sort the points by composition and then scan them from left to right while enforcing convexity.

**Algorithm idea**
- Sort all points by increasing $x$
- Start with an empty list called hull.
- Add points one by one from left to right.
- Every time a new point is added, check the last three points in the hull.
- If these three points form a concave bend, remove the middle point.
- Repeat this check until the last three points form a convex bend.
- Continue until all points have been processed.

The remaining points form the lower hull, or lower envelope.

**Convexity test**

For three consecutive points $P_2=(x_2,y_2), P_1=(x_1,y_1),P_3=(x_3,y_3)$, we compute

$$cross=(x_2−x_1)(y_3−y_1)−(y_2−y_1)(x_3−x_1).$$

This quantity tells us the turning direction:

- If $cross > 0$, the three points make a left turn, which is convex and acceptable for the lower hull.
- If $cross \le 0$, the middle point lies on or above the line connecting the other two points, so it is removed

**Why this works**

The lower envelope must be convex. If three consecutive points make a concave bend, then the middle point cannot belong to the lower convex boundary, because a straight line connecting the two outer points lies below it. Removing such points leaves only the points that define the lowest convex boundary.

In [ ]:
def lower_hull(points):
    points = points[np.argsort(points[:, 0])]
    hull = []

    for p in points:
        while len(hull) >= 2:
            x1, y1 = hull[-2]
            x2, y2 = hull[-1]
            x3, y3 = p
            cross = (x2 - x1)*(y3 - y1) - (y2 - y1)*(x3 - x1)
            if cross <= 0:
                hull.pop()
            else:
                break
        hull.append(p)

    return np.array(hull)

**Test this `lower_hull` function on a set of random points.**

In [ ]:
np.random.seed(42)
x_rnd = np.random.rand(40)
y_rnd = np.random.rand(40)

p_rnd = np.column_stack([x_rnd, y_rnd])
lh_rnd = lower_hull(p_rnd)


fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(x_rnd, y_rnd, label='Random scatter')
ax.plot(lh_rnd[:, 0], lh_rnd[:, 1], 'r-', lw=2, label='Lower envelope')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Random Scatter with Lower Convex Envelope')
ax.legend()
plt.show()


---
Let's get back to common tangent construction.

In [ ]:
lh = lower_hull(pts)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(xt1, yt1, label='scatter points')
ax.plot(lh[:, 0], lh[:, 1], 'r-', lw=2, label='Lower envelope')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Scatter points with Lower Convex Envelope')
ax.legend()
plt.show()

### How to find the common tangent line segment?

Let's find the longest segment of the lower envelop.

In [ ]:
# Find the longest lower-hull segment
dx = lh[1:, 0] - lh[:-1, 0]
i_tan = np.argmax(dx)

p1 = lh[i_tan]
p2 = lh[i_tan + 1]

print("Common tangent endpoints:")
print("x1, y1 =", p1)
print("x2, y2 =", p2)

# line equation: y = m x + b
m = (p2[1] - p1[1]) / (p2[0] - p1[0])
b = p1[1] - m * p1[0]

print("Common tangent: y = %.6f x + %.6f" % (m, b))

# plot
xx = np.linspace(p1[0], p2[0], 200)
yy = m * xx + b

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(xt1, yt1, label='scatter')
ax.plot(lh[:, 0], lh[:, 1], 'r-', lw=2, label='lower hull')
# ax.plot(xx, yy, 'k--', lw=3, label='common tangent')
ax.scatter([p1[0], p2[0]], [p1[1], p2[1]], c='k', s=60)
ax.legend()
plt.show()

&#9989; Do This -- Why the longest segment of the lower envelope is the common tangent line? Answer the question using the [link](https://forms.office.com/r/skdxChf2jp) here.


---
### Full convex hull

Here is an example of find close envelop of a set of data points.

In [ ]:
from scipy.spatial import ConvexHull

hull = ConvexHull(p_rnd)

# vertices of hull
hv = hull.vertices
hull_pts = p_rnd[hv]
# print(hull_pts)

# sort hull points by x
hull_pts = hull_pts[np.argsort(hull_pts[:, 0])]



In [ ]:
# # plot everything
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(x_rnd, y_rnd, label='Points')

# full hull
closed = np.vstack([p_rnd[hull.vertices], p_rnd[hull.vertices][0]])
ax.plot(closed[:, 0], closed[:, 1], 'k--', lw=1.5, label='Full hull')

ax.set_title('Convex Hull Envelope')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend()
plt.show()

&#9989; Do This -- How the outer convex hull is found? Answer using this [link](https://forms.office.com/r/skdxChf2jp).